# Arle 51-Language Training (Colab Free)
**Arlecchino IDE — Universal Code Completion Model**

## Architecture
- **Model**: 2-layer Bi-LSTM + Attention (~10M params)
- **Vocab**: 32K BPE tokens (universal)
- **Output**: ~15-20MB ONNX (INT8 quantized)

## Training Plan (3 Sessions, ~4-5h total)
| Session | Languages | Samples | Time |
|---------|-----------|---------|------|
| 1 | Tier 1 (15 langs) | 300K | ~2h |
| 2 | Tier 2 (12 langs) | 240K | ~1.5h |
| 3 | Tier 3 + Data (22 langs) | 200K | ~1h |

## Important Files to Save Between Sessions
- `arle_51lang_checkpoint.pt` — model weights
- `arle_tokenizer_51lang.json` — tokenizer (created in Session 1 ONLY)

## Step 0: Enable GPU & Check Resources

**IMPORTANT:** `Runtime` → `Change runtime type` → `T4 GPU` → `Save`

In [ ]:
import torch
import psutil
import os

print("=" * 50)
print("RESOURCE CHECK")
print("=" * 50)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("❌ No GPU! Enable T4 in Runtime → Change runtime type")

ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1e9:.1f} GB total, {ram.available / 1e9:.1f} GB available")
print(f"Disk: {psutil.disk_usage('/').free / 1e9:.1f} GB free")
print("=" * 50)
print("✅ Ready for training!")

## Step 1: Install Dependencies

In [ ]:
!pip install -q datasets tokenizers onnx onnxruntime tqdm psutil huggingface_hub

## Step 1.5: HuggingFace Login (REQUIRED for gated datasets)

**`bigcode/the-stack-smol` is a gated dataset.**

Steps:
1. Go to https://huggingface.co/datasets/bigcode/the-stack-smol
2. Click "Agree and access repository"
3. Create token at https://huggingface.co/settings/tokens (read access)
4. Run cell below and paste token

In [ ]:
from huggingface_hub import login

# Option 1: Interactive login (will prompt for token)
login()

# Option 2: Direct token (uncomment and paste your token)
# login(token="hf_your_token_here")

print("✅ HuggingFace authentication complete!")

## Step 2: Configure Session

**Change SESSION number for each training session:**
- Session 1: Train on Tier 1 (15 critical languages)
- Session 2: Fine-tune on Tier 2 (12 important languages)
- Session 3: Fine-tune on Tier 3 + Data (22 niche languages)

In [ ]:
# ============================================
# 🔧 CONFIGURATION — CHANGE THIS EACH SESSION
# ============================================
SESSION = 1  # Change to 2 or 3 for subsequent sessions

# Language tiers (Stack Overflow 2025 survey)
TIER_1 = [
    'Python', 'JavaScript', 'TypeScript', 'Java', 'C#',
    'C++', 'C', 'Go', 'Rust', 'PHP',
    'Ruby', 'Swift', 'Kotlin', 'Scala', 'Shell'
]

TIER_2 = [
    'Lua', 'R', 'Perl', 'Haskell', 'Clojure',
    'Elixir', 'Erlang', 'Julia', 'Dart', 'Groovy',
    'PowerShell', 'Objective-C'
]

TIER_3 = [
    'F#', 'OCaml', 'Fortran', 'COBOL', 'Ada',
    'Prolog', 'Lisp', 'Scheme', 'Racket', 'Common Lisp',
    'Assembly', 'VHDL', 'Verilog', 'Zig', 'Nim', 'Crystal'
]

DATA_CONFIG = [
    'JSON', 'YAML', 'TOML', 'XML', 'Markdown', 'Dockerfile'
]

# Session configuration
if SESSION == 1:
    TARGET_LANGS = TIER_1
    MAX_SAMPLES_PER_LANG = 20000  # 15 × 20K = 300K total
    EPOCHS = 3
    LEARNING_RATE = 1e-3
    print(f"📚 Session 1: Training on {len(TARGET_LANGS)} Tier 1 languages")
elif SESSION == 2:
    TARGET_LANGS = TIER_2
    MAX_SAMPLES_PER_LANG = 20000  # 12 × 20K = 240K total
    EPOCHS = 2
    LEARNING_RATE = 5e-4  # Lower LR for fine-tuning
    print(f"📚 Session 2: Fine-tuning on {len(TARGET_LANGS)} Tier 2 languages")
else:
    TARGET_LANGS = TIER_3 + DATA_CONFIG
    MAX_SAMPLES_PER_LANG = 10000  # 22 × 10K = 220K total
    EPOCHS = 2
    LEARNING_RATE = 3e-4  # Even lower for final fine-tuning
    print(f"📚 Session 3: Fine-tuning on {len(TARGET_LANGS)} Tier 3 + Data languages")

print(f"Languages: {TARGET_LANGS}")
print(f"Max samples per language: {MAX_SAMPLES_PER_LANG:,}")
print(f"Estimated total: {len(TARGET_LANGS) * MAX_SAMPLES_PER_LANG:,} samples")

## Step 3: Load Dataset (Streaming)

Using `codeparrot/github-code-clean` — open dataset, no authentication required.

In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import random
from collections import Counter
import gc

MAX_FILE_SIZE = 8000  # Characters (smaller = faster training)
MIN_FILE_SIZE = 50

all_code = []

print("Loading codeparrot/github-code-clean (streaming)...")
print(f"Target: {MAX_SAMPLES_PER_LANG:,} samples per language\n")

# Load dataset in streaming mode
ds = load_dataset(
    "codeparrot/github-code-clean",
    streaming=True,
    split="train",
    trust_remote_code=True
)

# Track counts per language
lang_counts = {lang: 0 for lang in TARGET_LANGS}
target_set = set(TARGET_LANGS)

# Stream and collect
pbar = tqdm(ds, desc="Streaming data")
for item in pbar:
    lang = item.get('language', '')
    
    # Check if this language is in our target
    if lang not in target_set:
        continue
    
    # Check if we have enough for this language
    if lang_counts[lang] >= MAX_SAMPLES_PER_LANG:
        # Check if all languages are done
        if all(c >= MAX_SAMPLES_PER_LANG for c in lang_counts.values()):
            break
        continue
    
    content = item.get('code', '')
    
    # Filter by size
    if len(content) < MIN_FILE_SIZE or len(content) > MAX_FILE_SIZE:
        continue
    
    all_code.append({
        'code': content,
        'language': lang
    })
    lang_counts[lang] += 1
    
    # Update progress
    total = sum(lang_counts.values())
    pbar.set_postfix({'collected': f'{total:,}'})

# Shuffle
random.shuffle(all_code)

print(f"\n✅ Total samples: {len(all_code):,}")
print("\nLanguage distribution:")
for lang, count in sorted(lang_counts.items(), key=lambda x: -x[1]):
    if count > 0:
        print(f"  {lang}: {count:,}")

# Clear memory
gc.collect()

## Step 4: Build/Load Tokenizer

**Session 1**: Build new tokenizer from data  
**Session 2-3**: Load existing tokenizer (upload `arle_tokenizer_51lang.json`)

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors

VOCAB_SIZE = 32000
TOKENIZER_FILE = "arle_tokenizer_51lang.json"

if SESSION == 1:
    print("Building NEW BPE tokenizer (Session 1)...")
    
    tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    
    special_tokens = ["<PAD>", "<UNK>", "<BOS>", "<EOS>", "<MASK>"]
    
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=special_tokens,
        show_progress=True,
        min_frequency=2
    )
    
    def code_iterator():
        for item in all_code:
            yield item['code']
    
    tokenizer.train_from_iterator(code_iterator(), trainer=trainer)
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
    
    tokenizer.save(TOKENIZER_FILE)
    print(f"✅ Tokenizer saved: {TOKENIZER_FILE}")
else:
    if not os.path.exists(TOKENIZER_FILE):
        raise FileNotFoundError(
            f"❌ {TOKENIZER_FILE} not found!\n"
            "Upload it from Session 1 before running Session 2-3."
        )
    print(f"Loading existing tokenizer from {TOKENIZER_FILE}")
    tokenizer = Tokenizer.from_file(TOKENIZER_FILE)

print(f"Vocab size: {tokenizer.get_vocab_size():,}")

# Test tokenization
test_samples = [
    "def hello():\n    print('Hello')",
    "function hello() { console.log('Hello'); }",
    "func main() { fmt.Println(\"Hello\") }"
]
print("\nTokenization test:")
for sample in test_samples:
    tokens = tokenizer.encode(sample).tokens[:8]
    print(f"  '{sample[:35]}...' → {tokens}")

## Step 5: Prepare PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

MAX_SEQ_LEN = 256  # Reduced for memory efficiency
BATCH_SIZE = 48    # Optimized for T4 16GB

class CodeDataset(Dataset):
    def __init__(self, code_samples, tokenizer, max_len=MAX_SEQ_LEN):
        self.samples = []
        self.max_len = max_len
        
        print("Tokenizing samples...")
        for item in tqdm(code_samples):
            try:
                encoded = tokenizer.encode(item['code'])
                ids = encoded.ids
                
                # Need at least 10 tokens for meaningful training
                if len(ids) >= 10:
                    # Truncate if needed
                    if len(ids) > max_len:
                        ids = ids[:max_len]
                    self.samples.append(ids)
            except Exception:
                continue
        
        print(f"✅ Created {len(self.samples):,} training samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        ids = self.samples[idx]
        
        # Input: all tokens except last
        # Target: all tokens except first (predict next token)
        input_ids = ids[:-1]
        target_ids = ids[1:]
        
        # Pad to max_len - 1
        pad_len = self.max_len - 1 - len(input_ids)
        if pad_len > 0:
            input_ids = input_ids + [0] * pad_len
            target_ids = target_ids + [0] * pad_len
        
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'target_ids': torch.tensor(target_ids, dtype=torch.long)
        }

# Create dataset
dataset = CodeDataset(all_code, tokenizer)

# Free memory from raw code
del all_code
gc.collect()

# Train/val split
train_size = int(0.95 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"Train: {len(train_dataset):,}, Val: {len(val_dataset):,}")

# Create data loaders (NO num_workers for Colab stability)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Step 6: Define Model

In [ ]:
import torch.nn as nn

class ArleLSTM(nn.Module):
    """Arle: 2-layer Bi-LSTM with Attention for code completion."""
    
    def __init__(self, vocab_size=32000, embed_dim=128, hidden_dim=256, 
                 num_layers=2, dropout=0.1, max_seq_len=256):
        super().__init__()
        
        self.max_seq_len = max_seq_len
        
        # Embeddings
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        
        # Bi-LSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Self-attention
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        
        # Output heads
        self.next_token_head = nn.Linear(hidden_dim * 2, vocab_size)
        
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() > 1:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
    
    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        
        # Position indices
        positions = torch.arange(seq_len, device=input_ids.device)
        positions = positions.unsqueeze(0).expand(batch_size, -1)
        
        # Embeddings
        x = self.embedding(input_ids) + self.pos_embedding(positions)
        x = self.dropout(x)
        
        # LSTM
        lstm_out, _ = self.lstm(x)
        
        # Self-attention with residual
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        x = self.layer_norm(lstm_out + attn_out)
        x = self.dropout(x)
        
        # Output logits
        logits = self.next_token_head(x)
        
        return logits

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Checkpoint file
CHECKPOINT_FILE = "arle_51lang_checkpoint.pt"

# Load or create model
if SESSION > 1 and os.path.exists(CHECKPOINT_FILE):
    print(f"\nLoading checkpoint from {CHECKPOINT_FILE}")
    checkpoint = torch.load(CHECKPOINT_FILE, map_location=device)
    
    model = ArleLSTM(
        vocab_size=tokenizer.get_vocab_size(),
        max_seq_len=MAX_SEQ_LEN
    ).to(device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    start_epoch = checkpoint.get('epoch', 0) + 1
    prev_langs = checkpoint.get('languages', [])
    print(f"✅ Loaded model trained on: {prev_langs}")
    print(f"Resuming from epoch {start_epoch}")
else:
    if SESSION > 1:
        print(f"⚠️ No checkpoint found. Starting fresh model.")
    
    model = ArleLSTM(
        vocab_size=tokenizer.get_vocab_size(),
        max_seq_len=MAX_SEQ_LEN
    ).to(device)
    start_epoch = 0

# Model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters: {total_params:,}")
print(f"Trainable: {trainable_params:,}")
print(f"Estimated size: ~{total_params * 4 / 1e6:.1f} MB (FP32)")

## Step 7: Training Loop

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    
    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        target_ids = batch['target_ids'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids)
        
        loss = criterion(
            logits.view(-1, logits.size(-1)),
            target_ids.view(-1)
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(loader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        target_ids = batch['target_ids'].to(device)
        
        logits = model(input_ids)
        loss = criterion(
            logits.view(-1, logits.size(-1)),
            target_ids.view(-1)
        )
        total_loss += loss.item()
        
        # Accuracy (ignoring padding)
        preds = logits.argmax(dim=-1)
        mask = target_ids != 0
        correct += ((preds == target_ids) & mask).sum().item()
        total += mask.sum().item()
    
    accuracy = correct / total if total > 0 else 0
    return total_loss / len(loader), accuracy

# Training
print("\n" + "=" * 60)
print(f"SESSION {SESSION}: Training on {device}")
print(f"Languages: {TARGET_LANGS}")
print(f"Epochs: {EPOCHS}, LR: {LEARNING_RATE}")
print("=" * 60)

best_val_loss = float('inf')
training_start = time.time()
all_trained_langs = TARGET_LANGS.copy()

# Add previous languages if continuing
if SESSION > 1 and 'prev_langs' in dir():
    all_trained_langs = list(set(prev_langs + TARGET_LANGS))

for epoch in range(EPOCHS):
    epoch_num = start_epoch + epoch + 1
    print(f"\n--- Epoch {epoch + 1}/{EPOCHS} (Global: {epoch_num}) ---")
    
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%}")
    
    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch_num,
            'session': SESSION,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
            'languages': all_trained_langs,
            'vocab_size': tokenizer.get_vocab_size()
        }, CHECKPOINT_FILE)
        print(f"  💾 Saved checkpoint (val_loss: {val_loss:.4f})")

elapsed = time.time() - training_start
print(f"\n✅ Session {SESSION} completed in {elapsed/60:.1f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")

## Step 8: Export to ONNX

**Run this after Session 3** (or whenever you want a usable model)

In [ ]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

# Load best checkpoint
checkpoint = torch.load(CHECKPOINT_FILE, map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
model.cpu()

print(f"Model trained on: {checkpoint.get('languages', 'unknown')}")
print(f"Val loss: {checkpoint['val_loss']:.4f}, Val acc: {checkpoint['val_acc']:.2%}")

# Export to ONNX
ONNX_FILE = "arle_51lang.onnx"
ONNX_INT8_FILE = "arle_51lang_int8.onnx"

dummy_input = torch.randint(0, 1000, (1, 128))  # Batch=1, SeqLen=128

print("\nExporting to ONNX...")
torch.onnx.export(
    model,
    dummy_input,
    ONNX_FILE,
    input_names=['input_ids'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch', 1: 'seq_len'},
        'logits': {0: 'batch', 1: 'seq_len'}
    },
    opset_version=14,
    do_constant_folding=True
)

# Verify
onnx_model = onnx.load(ONNX_FILE)
onnx.checker.check_model(onnx_model)
print("✅ ONNX model verified")

# Quantize to INT8
print("Quantizing to INT8...")
quantize_dynamic(
    ONNX_FILE,
    ONNX_INT8_FILE,
    weight_type=QuantType.QUInt8
)

# File sizes
onnx_size = os.path.getsize(ONNX_FILE) / 1e6
int8_size = os.path.getsize(ONNX_INT8_FILE) / 1e6
print(f"\n📦 ONNX: {onnx_size:.1f} MB → INT8: {int8_size:.1f} MB")

## Step 9: Test Model

In [ ]:
import onnxruntime as ort
import numpy as np

# Load ONNX model
session = ort.InferenceSession(ONNX_INT8_FILE)

def predict(code_prefix, num_tokens=15, temperature=0.8):
    """Generate code completion with temperature sampling."""
    encoded = tokenizer.encode(code_prefix)
    input_ids = np.array([encoded.ids], dtype=np.int64)
    
    predictions = []
    for _ in range(num_tokens):
        outputs = session.run(None, {'input_ids': input_ids})
        logits = outputs[0][0, -1, :]
        
        # Temperature sampling
        if temperature > 0:
            logits = logits / temperature
            probs = np.exp(logits) / np.sum(np.exp(logits))
            next_token_id = np.random.choice(len(probs), p=probs)
        else:
            next_token_id = int(np.argmax(logits))
        
        # Stop on special tokens
        if next_token_id in [0, 1, 3]:  # PAD, UNK, EOS
            break
        
        predictions.append(tokenizer.decode([next_token_id]))
        input_ids = np.concatenate([input_ids, [[next_token_id]]], axis=1)
    
    return ''.join(predictions)

# Test various languages
tests = [
    ("Python", "def calculate_total("),
    ("JavaScript", "function handleClick("),
    ("Go", "func GetUser("),
    ("Rust", "fn process_data("),
    ("Java", "public void save"),
    ("PHP", "public function "),
    ("TypeScript", "interface User"),
    ("C#", "public async Task<"),
]

print("🧪 Testing predictions:\n")
for lang, prefix in tests:
    completion = predict(prefix, num_tokens=20)
    print(f"{lang}:")
    print(f"  Input:  '{prefix}'")
    print(f"  Output: '{completion}'")
    print()

## Step 10: Download Files

**After training:**
- Download `arle_51lang_int8.onnx` → `~/.arlecchino/models/`
- Download `arle_tokenizer_51lang.json` → `~/.arlecchino/models/`

**Between sessions:**
- Download `arle_51lang_checkpoint.pt` and `arle_tokenizer_51lang.json`
- Upload them before running next session

In [ ]:
from google.colab import files

print("📥 Downloading files...\n")

# Always download checkpoint and tokenizer for next session
if os.path.exists(CHECKPOINT_FILE):
    files.download(CHECKPOINT_FILE)
    print(f"✅ {CHECKPOINT_FILE}")

if os.path.exists(TOKENIZER_FILE):
    files.download(TOKENIZER_FILE)
    print(f"✅ {TOKENIZER_FILE}")

# Download ONNX if exported
if os.path.exists(ONNX_INT8_FILE):
    files.download(ONNX_INT8_FILE)
    print(f"✅ {ONNX_INT8_FILE}")

print("\n" + "=" * 60)
print("INSTALLATION:")
print("Place these files in ~/.arlecchino/models/")
print("  - arle_51lang_int8.onnx")
print("  - arle_tokenizer_51lang.json")
print("=" * 60)

## 📋 Session Checklist

### Session 1 (First Time)
1. ✅ Set `SESSION = 1`
2. ✅ Run all cells (Step 0 → Step 7)
3. ✅ Download: `checkpoint.pt` + `tokenizer.json`
4. ⏸️ Optional: Export ONNX (Step 8-9) if you want early model

### Session 2
1. 📤 Upload: `checkpoint.pt` + `tokenizer.json`
2. ✅ Set `SESSION = 2`
3. ✅ Run all cells
4. ✅ Download updated checkpoint

### Session 3 (Final)
1. 📤 Upload: `checkpoint.pt` + `tokenizer.json`
2. ✅ Set `SESSION = 3`
3. ✅ Run all cells
4. ✅ Export ONNX (Step 8)
5. ✅ Test model (Step 9)
6. ✅ Download: `arle_51lang_int8.onnx` + `tokenizer.json`